# Classifying the Changing Sky with FLARE

### A hands-on lesson in machine learning for astronomy

Every clear night, robotic telescopes such as the **Zwicky Transient Facility (ZTF)**
photograph the sky again and again, looking for things that *change*: stars that
explode, black holes that flare up, stars being torn apart. Astronomers call these
**transients**, and each night brings far too many of them for people to check by hand.

In this notebook we build up, step by step, the machine-learning system called
**FLARE** (*Feature-based Light-curve Aggregated Ranking Ensemble*). FLARE classifies
six kinds of objects from their **light curves** (brightness over time):

| Class | What it is | How its light behaves |
|---|---|---|
| **SN_Ia** | Type Ia supernova: a white dwarf explodes | fast rise, smooth decline, turns red |
| **SN_CC** | Core-collapse supernova: a massive star collapses (types II, Ib, Ic) | slower, often a long "plateau" |
| **SLSN** | Superluminous supernova | very bright, rises and fades over months |
| **AGN** | Active galactic nucleus: a feeding supermassive black hole | flickers randomly for years |
| **TDE** | Tidal disruption event: a star torn apart by a black hole | smooth decline, stays **blue** |
| **CV** | Cataclysmic variable: a white dwarf pulling gas from a companion star | repeated outbursts |

All the data come from the benchmark that ships inside the FLARE repository: about ten
thousand ZTF objects from the Bright Transient Survey, each with a spectroscopic
classification, already split into training, validation and test sets.

**The plan:**
1. Look at real light curves
2. Turn curves into *numbers with a physical meaning* (features)
3. Let decision trees learn from the features
4. Ask the model *how sure* it is, and check whether that confidence is honest
5. Ask the model *why*, and check that it learned real physics

> **Requirements:** the `flare` package (`pip install -e ..` from this folder's parent)
> plus `matplotlib` and `lightgbm`. The first run unpacks the benchmark light curves
> (`benchmark/light_curves.tar.gz`) once.

In [ ]:
# --- Setup ---------------------------------------------------------------
import sys, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")            # find the flare package if not pip-installed
import flare
from flare import benchmark as B
from flare import data as D
from flare.features import extract_from_array

B.prepare()                         # unpacks the light curves once; a no-op afterwards
CLASSES = B.SIX                     # ['SN_Ia', 'SN_CC', 'SLSN', 'AGN', 'TDE', 'CV']

# one colour per class, fixed everywhere: six shades of matplotlib's magma map
# (we stop at 0.85 to avoid the near-white end)
CLASS_COLOR = dict(zip(CLASSES, plt.cm.magma(np.linspace(0.12, 0.85, len(CLASSES)))))
# the photometric bands are drawn in neutral greys so they never look like a class
BAND_STYLE = {"g": dict(color="0.10", marker="o"), "r": dict(color="0.50", marker="s"),
              "i": dict(color="0.75", marker="^")}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Setup OK, classes:", CLASSES)

## 1. Meet the data: real light curves

Each object is stored as a table of **detections**: when we saw it, in which filter
(**g** = green light, **r** = red light, sometimes **i** = near-infrared), and how bright
it was. Telescopes cannot observe every night (weather, moonlight, daytime), so the
sampling is **irregular**, which is one of the things that makes this problem hard.

The benchmark comes in four parts. We use the **training** split to explore and to fit
our own small models, and keep the **test** split for the final, honest evaluation.
(The fourth part, `anomaly`, holds rare objects outside the six classes.)

In [ ]:
# Load the training and test catalogues
X_train, y_train, train = B.load_split("train")
X_test,  y_test,  test  = B.load_split("test")

counts = pd.DataFrame({"train": train.cls.value_counts(),
                       "test": test.cls.value_counts()}).reindex(CLASSES)
print(counts.to_string())
print(f"\nTotal: {len(train)} training objects, {len(test)} test objects")

The classes are very unequal: type Ia supernovae make up 68% of the test set,
while there are only 10 TDEs. Keep this in mind, because it will matter when we
judge the model later.

Now let us plot one real example of each class, choosing the object with the most
detections in the training set.

In [ ]:
# Plot one example light curve per class
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, cls in zip(axes.ravel(), CLASSES):
    # pick the object of this class with the most detections (the clearest curve)
    row = train[train.cls == cls].nlargest(1, "n_events").iloc[0]
    arr = D.load_events(str(B.BENCHMARK / row.filepath))   # (n_events, 15), first 100 days
    bands = D.reconstruct_bands(arr)                         # per-band light curves
    for name, lc in bands.items():
        ax.errorbar(lc.t, lc.logflux, yerr=lc.logflux_err, fmt=BAND_STYLE[name]["marker"],
                    ms=3, color=BAND_STYLE[name]["color"], alpha=0.8, label=f"{name} band")
    ax.set_title(f"{cls}  ({row.obj_id})", color=CLASS_COLOR[cls], fontweight="bold")
    ax.set_xlabel("days since first detection")
for ax in axes[:, 0]:
    ax.set_ylabel(r"brightness  ($\log_{10}$ flux)")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Look closely**, because you can already *see* the physics:

* The **supernovae** rise and then fade away: they are explosions, one-time events.
  The superluminous one evolves much more slowly than the ordinary ones.
* The **AGN** wanders up and down; it never "ends", it just flickers.
* The **CV** jumps up and down in outbursts.
* For the **TDE**, the green (g) points sit *above* the red (r) points for the whole
  curve: it stays **blue and hot** while the black hole feeds on the star.

Our whole job is to teach a computer to notice the same things we just noticed.

### Exercise 1
Change the code above to plot a *different* object of each class
(hint: replace `.nlargest(1, "n_events")` with `.sample(1)` and re-run).
Do the patterns hold?

## 2. From pictures to numbers: feature engineering

A computer cannot "look" at a plot. We must convert each light curve into a fixed
list of **numbers that capture the physics**, called **features**.
FLARE computes **160** of them per light curve. A few examples:

| Feature | What it measures | Which objects it helps with |
|---|---|---|
| `g_r_mean` | average colour (green minus red brightness, in magnitudes) | TDEs are **blue**, so very negative |
| `g_r_late_minus_early` | does the colour *change* with time? | supernovae turn red; TDEs do not |
| `g_eta_e` | is the curve smooth or jumpy? | AGN flicker; TDEs are smooth |
| `g_bazin_fall_time` | how many days the curve takes to fade | explosions fade; AGN do not |
| `g_bazin_reduced_chi2` | does an "explosion-shaped" fit work at all? | separates transients from AGN and CVs |

The `bazin` features come from fitting each curve with a simple rise-then-fall model
of an explosion (Bazin et al. 2009).

On top of the light curve, the shipped model also reads 30 **context** columns: the
host galaxy's colours and distance from the transient, a rough absolute magnitude from
the host's photometric redshift (`M_pseudo`), and whether there is a star at that
position in the Gaia, WISE and Pan-STARRS catalogues. All 190 columns come
precomputed in the benchmark, so `B.load_split` already returned them.

In [ ]:
# Extract the 160 light-curve features for ONE object and peek at a few
row = train[train.cls == "TDE"].nlargest(1, "n_events").iloc[0]
arr = D.load_events(str(B.BENCHMARK / row.filepath))
feats = extract_from_array(arr)
print(f"{row.obj_id} (a real TDE) -> {len(feats)} features. A physicist's sample:\n")
for k in ["g_r_mean", "g_r_late_minus_early", "g_eta_e", "g_bazin_fall_time", "g_bazin_reduced_chi2"]:
    print(f"  {k:24s} = {feats[k]: .3f}")

# the precomputed benchmark table holds exactly the same numbers
i = row.name
same = all(np.isclose(v, X_train.loc[i, k], rtol=1e-4) or (np.isnan(v) and np.isnan(X_train.loc[i, k]))
           for k, v in feats.items())
print(f"\nMatches the precomputed benchmark features: {same}")

In [ ]:
# Do these features actually separate the classes? Plot colour against smoothness
# for every object in the training set (rare classes drawn last, on top).
fig, ax = plt.subplots(figsize=(7.5, 5))
for cls in CLASSES:
    m = (train.cls == cls).to_numpy()
    rare = m.sum() < 500
    ax.scatter(X_train.g_r_mean[m], np.log10(X_train.g_eta_e[m].clip(1e-4)),
               s=14 if rare else 5, alpha=0.8 if rare else 0.25,
               color=CLASS_COLOR[cls], label=f"{cls} (n={m.sum()})", zorder=3 if rare else 1)
ax.set_xlabel("mean colour  g_r_mean   (left = bluer)")
ax.set_ylabel(r"smoothness  $\log_{10}$ g_eta_e   (up = jumpier)")
ax.set_xlim(-1.5, 2.0)
ax.set_title("Two features already separate part of the sky")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.show()

Two numbers, colour and smoothness, and some structure appears already: the TDEs and
CVs sit on the *blue* side, while most supernovae are redder. Smoothness on its own
helps less than we might hope, because objects of every class scatter up into the
jumpy region when their light curves are sparse or noisy. The two kinds of supernova
sit almost on top of each other, so we need something smarter than drawing lines by
hand.

### Exercise 2
Swap `g_eta_e` for `g_bazin_fall_time` in the plot above.
Which pair of classes does *that* feature separate best?

## 3. Let the trees decide

A **decision tree** asks yes/no questions about the features
(*"is `g_r_mean` < -0.2?"*) and routes each object down to a leaf with a class.
One tree is weak. **Gradient boosting** (we use **LightGBM**) builds *hundreds* of
small trees, each one correcting the mistakes of those before it, like a class of
students where each new student fixes the previous one's errors.

Why trees and not a deep neural network? For *tabular* data like our features,
boosted trees are very hard to beat, they train in **minutes on a laptop CPU**,
they handle missing values natively (no i band? no host galaxy? no problem), and,
importantly for science, **we can ask them why**.

Let us train a *tiny* 2-feature model on the training set to see the idea, then load
the real one.

In [ ]:
# A tiny 2-feature LightGBM, TDE vs AGN, just to visualise the decision boundary
import lightgbm as lgb
pair = train.cls.isin(["TDE", "AGN"]).to_numpy()
X2 = np.c_[X_train.g_r_mean[pair], np.log10(X_train.g_eta_e[pair].clip(1e-4))]
y2 = (train.cls[pair] == "TDE").astype(int).to_numpy()
mini = lgb.LGBMClassifier(n_estimators=60, num_leaves=7, min_child_samples=5,
                          class_weight="balanced", n_jobs=1, verbosity=-1).fit(X2, y2)

# paint the plane with the model's decision
ok = np.isfinite(X2).all(1)
gx = np.linspace(*np.percentile(X2[ok, 0], [1, 99]), 200)
gy = np.linspace(*np.percentile(X2[ok, 1], [1, 99]), 200)
GX, GY = np.meshgrid(gx, gy)
Z = mini.predict_proba(np.c_[GX.ravel(), GY.ravel()])[:, 1].reshape(GX.shape)

fig, ax = plt.subplots(figsize=(7.5, 5))
cf = ax.contourf(GX, GY, Z, levels=20, cmap="magma", alpha=0.35)
fig.colorbar(cf, ax=ax, label="P(TDE) from the tiny model")
for cls, edge in [("AGN", "white"), ("TDE", "black")]:
    s = train.cls[pair].to_numpy() == cls
    ax.scatter(X2[s, 0], X2[s, 1], s=22, color=CLASS_COLOR[cls], edgecolor=edge,
               linewidth=0.5, label=f"{cls} (n={s.sum()})")
ax.set_xlabel("g_r_mean (left = bluer)"); ax.set_ylabel(r"$\log_{10}$ g_eta_e (up = jumpier)")
ax.set_xlim(gx[0], gx[-1]); ax.set_ylim(gy[0], gy[-1])
ax.set_title("What 60 tiny trees learned: light region = 'TDE', dark = 'AGN'")
ax.legend()
plt.tight_layout(); plt.show()

The trees carved the plane into regions, and the boundary is *not* a straight line:
it bends around the data. The real FLARE does exactly this, but with **190 columns and
814 (705 + 109) trees** in total.

## 4. The real FLARE model

The trained model ships with the package as plain LightGBM text files. It has two
parts:

* a **top-level** model that chooses between five groups: SN_Ia, a combined
  "core-collapse or superluminous" group, AGN, TDE and CV;
* a small **SLSN branch** that splits that combined group into SN_CC and SLSN.

Multiplying the two gives probabilities over the six classes. The shipped model was
trained on the training and validation splits, so the **test split holds objects it
has never seen**. Let us classify one test object of each class.

In [ ]:
clf = flare.load_classifier()        # the six-class (bts6) model
print(f"Loaded: {clf.top.num_trees()} top-level trees + {clf.branch.num_trees()} SLSN-branch trees, "
      f"{len(clf.feature_names)} input columns")

Xc = X_test.reindex(columns=clf.feature_names)   # the column order the model expects

# one test object per class: the one with the most detections
picks = [test[test.cls == c].nlargest(1, "n_events").index[0] for c in CLASSES]
proba = clf.predict_proba(Xc.loc[picks])

df = pd.DataFrame(proba, columns=CLASSES).round(3)
df.insert(0, "prediction", clf.predict(Xc.loc[picks]))
df.insert(0, "truth", test.cls[picks].to_numpy())
df.insert(0, "object", test.obj_id[picks].to_numpy())
df

Each row is a probability distribution over the six classes. Five of the six
objects are classified correctly. The TDE ZTF26aaoebni is called SN_CC with probability
0.973, a reminder that a confident answer can still be wrong, and the reason we
measure the model on the whole test set rather than on a few examples.

Now the honest question every scientist must ask:

## 5. Can we *trust* those probabilities?

If a model says "90%" for a thousand objects, then roughly **900 of them should
actually be that class**. When that holds, the model is **well-calibrated**.
Many powerful models are *overconfident*: they say "99%" and are wrong far more
than 1% of the time.

Let us measure this ourselves on the full test set. First the plain scores, then a
**reliability diagram**: we bin the predictions by confidence and check the actual
accuracy in each bin.

In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report

P = clf.predict_proba(Xc)
pred = np.array([CLASSES.index(c) for c in clf.predict(Xc)])
conf = P[np.arange(len(P)), pred]          # probability of the predicted class
y_true = y_test

print(f"N = {len(y_true)} unseen objects")
print(f"  accuracy          : {(pred == y_true).mean():.3f}")
print(f"  balanced accuracy : {balanced_accuracy_score(y_true, pred):.3f}  (mean recall over the six classes)")
print(f"  macro F1          : {f1_score(y_true, pred, average='macro'):.3f}\n")
print(classification_report(y_true, pred, target_names=CLASSES, digits=3))

The overall accuracy is 0.926, but remember that 68% of the test set are type Ia
supernovae: a model that always answered "SN_Ia" would already score 0.682. The
**balanced accuracy** (0.843) and **macro F1** (0.847) give every class the same weight,
so they are the fairer summary. The per-class table shows where the difficulty lies:
type Ia supernovae and CVs are recognised well (F1 of 0.959 and 0.929), while the superluminous supernovae are the hardest class (F1 of 0.642, precision 0.586). Most of their errors are mix-ups with ordinary core-collapse supernovae, in both directions, which makes sense: both are explosions of massive stars, and a light curve alone often cannot tell how bright the explosion really was.

In [ ]:
# Reliability diagram: does "x% confident" mean "x% correct"?
bins = np.linspace(0.2, 1.0, 9)
mids, accs, sizes = [], [], []
ece = 0.0
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf > lo) & (conf <= hi)
    if m.sum() == 0: continue
    ece += m.mean() * abs((pred[m] == y_true[m]).mean() - conf[m].mean())
    if m.sum() < 5: continue
    mids.append(conf[m].mean()); accs.append((pred[m] == y_true[m]).mean()); sizes.append(m.sum())

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot([0, 1], [0, 1], "--", color="0.4", label="perfect calibration")
ax.plot(mids, accs, "-o", color=CLASS_COLOR["SLSN"], label="FLARE")
for x0, a, n in zip(mids, accs, sizes):
    ax.annotate(f"n={n}", (x0, a), textcoords="offset points", xytext=(0, 8), fontsize=7)
ax.set_xlabel("model confidence"); ax.set_ylabel("actual accuracy")
ax.set_title("Reliability: does FLARE mean what it says?")
ax.legend(); ax.set_xlim(0.2, 1.02); ax.set_ylim(0.2, 1.02)
plt.tight_layout(); plt.show()
print(f"Expected calibration error (ECE): {ece:.3f}")
print(f"Share of objects with confidence above 0.9: {(conf > 0.9).mean():.1%}")

The curve stays close to the diagonal, and the expected calibration error (the average gap between stated confidence and actual accuracy, weighted by the number of objects in each bin) is 0.008, below one percentage point. Most objects (78.5%) receive a confidence above 0.9, and in that bin the model is right about 98% of the time. This is what lets a telescope scheduler act on these numbers.

## 6. Knowing what we don't know: conformal prediction

Sometimes one probability is not enough. For a rare TDE candidate we want a
**guarantee**. **Conformal prediction** turns probabilities into a
**prediction set**: a shortlist of classes built so that it contains the true class
about 90% of the time, separately **for every class** (so the promise also holds for
the rare TDEs, not only for the plentiful supernovae). The 90% is a promise about the
average over many objects; on a finite test set the measured rate wobbles around it,
and the wobble is largest for the smallest classes.

* Easy object: a set of one, `['AGN']`, and we are done.
* Genuinely ambiguous object: a set of two, `['SN_CC', 'TDE']`, which says "this one
  deserves a spectrum".

In [ ]:
sets = clf.prediction_sets(Xc)               # alpha = 0.1, i.e. a 90% target
set_sizes = np.array([len(s) for s in sets])
covered = np.array([CLASSES[t] in s for t, s in zip(y_true, sets)])
print(f"Average shortlist size : {set_sizes.mean():.2f} classes")
print(f"Truth inside shortlist : {covered.mean():.1%}  (target 90%)")
print(f"Empty shortlists       : {(set_sizes == 0).sum()}   (the object looks like no class)")
excl = np.array([len(s) > 0 and CLASSES[p] not in s for p, s in zip(pred, sets)])
print(f"Non-empty shortlists without the top guess: {excl.sum()}\n")

per_class = pd.DataFrame({"n": [(y_true == k).sum() for k in range(6)],
                          "coverage": [covered[y_true == k].mean() for k in range(6)],
                          "mean set size": [set_sizes[y_true == k].mean() for k in range(6)]},
                         index=CLASSES).round(3)
print(per_class.to_string(), "\n")

# show a few interesting (ambiguous) cases
amb = np.where(set_sizes >= 2)[0][:6]
for i in amb:
    print(f"  {test.obj_id[i]}: truth={CLASSES[y_true[i]]:6s} top guess={CLASSES[pred[i]]:6s} shortlist={sets[i]}")

In [ ]:
# Every prediction can carry its own error statement
i = amb[0]
print(f"{test.obj_id[i]}, truth = {CLASSES[y_true[i]]}")
rep = clf.conformal_report(Xc.iloc[[i]])[0]
for k in ["predicted", "set", "alpha", "p_values", "credibility", "confidence"]:
    print(f"{k:12s}: {rep[k]}")

On the test set the shortlists contain the true class 89.7% of the time with an average
size of only 1.04 classes: 1435 objects get a single answer, 97 get two or more
classes, and 43 get an empty shortlist because they look unusual for every class. The per-class coverage ranges from 0.844 (CV) to 0.958 (SLSN); with only
10 TDEs and 24 SLSNe in the test set, a single object moves those
numbers by several percentage points.

A shortlist does not have to contain the model's top guess. Each class has its own
threshold, set by how confident the model usually is on that class, and type Ia
supernovae are normally recognised with very high probability. So an object called
SN_Ia with a modest probability can fail the SN_Ia threshold while passing others, as in
the first example above; this happens for 63 objects with a non-empty shortlist.

The report above shows the **p-value** of each class: how typical this object would be
if it really belonged to that class. A class enters the shortlist when its p-value is
above 0.1. A low **credibility** (the largest p-value) means the object looks like
none of the six classes, which is a first hint that it might be something new.

### Exercise 3
Find the objects whose shortlist contains `TDE` but whose top guess is *not* TDE.
These are exactly the objects a TDE hunter would send to a big telescope.
How many are there, and how many of them really are TDEs?

## 7. Why did it say TDE? Opening the box

Because FLARE is made of decision trees over *physical* features, we can rank which
features its decisions rely on. If the model has learned real astrophysics, the
top features should match what we saw in Section 1 with our own eyes.

In [ ]:
imp = pd.Series(clf.top.feature_importance(importance_type="gain"), index=clf.feature_names)
imp = imp / imp.sum()                         # share of the total gain
lc_names = set(feats.keys())                  # the 160 light-curve features
top15 = imp.nlargest(15)[::-1]
colors = [CLASS_COLOR["SLSN"] if n in lc_names else CLASS_COLOR["TDE"] for n in top15.index]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top15.index, top15.values, color=colors, height=0.6)
ax.barh([], [], color=CLASS_COLOR["SLSN"], label="light-curve feature")
ax.barh([], [], color=CLASS_COLOR["TDE"], label="context column")
ax.set_xlabel("share of total gain (top-level model)")
ax.set_title("What FLARE actually pays attention to")
ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print("Top five:", ", ".join(top15.index[-5:][::-1]))
print(f"Share of the gain from the 160 light-curve features: {imp[list(lc_names)].sum():.1%}")
print(f"Share of the gain from the 30 context columns      : {imp.drop(list(lc_names)).sum():.1%}")

The top features are the slope of the r-band brightness (`r_linear_fit_slope`, how fast the object fades), the colour (`g_r_wmean` and `g_r_late`), the proper motion of a Gaia star at the same position (`pm`, which flags stars in our own Galaxy, such as CVs), and the time of peak brightness in r (`r_t_peak`). Rise and fade, colour, and "is this a star nearby?" are the same clues we spotted by eye in Section 1. The light-curve features carry 76.4% of the total gain and the context columns the remaining 23.6%. The machine did not learn magic; it learned astronomy.

## 8. What we learned

1. **Transients** tell their story through light curves, and the classes differ
   in *colour, smoothness and shape*.
2. **Feature engineering** turns physics into numbers a computer can use.
3. **Gradient-boosted trees** are simple, fast and excellent on this kind of data;
   we do not always need a giant neural network.
4. **Imbalanced classes** make plain accuracy misleading, so we also report balanced
   accuracy and macro F1.
5. **Calibration** is what makes a probability *mean something*.
6. **Conformal prediction** turns predictions into shortlists with a stated coverage
   target, set separately for every class, including the rare ones.
7. **Interpretability** closes the loop: the model's reasons match the physics.

### Final challenge
A telescope scheduler cannot wait 100 days. FLARE ships "early" versions trained on
only the **first 5, 10 or 20 days** of each light curve. Load the 10-day model with
`flare.load_classifier(horizon_days=10)`, recompute the light-curve features on the
truncated curves with
`clf10.features_from_files([str(B.BENCHMARK / p) for p in test.filepath], horizon_days=10)`
(this takes about ten seconds), and re-run Section 5.
Which classes survive early classification, and which become nearly impossible? Why?
(The context columns are left empty here; the model was trained to cope with that.)

### Bonus challenge
`B.load_split("anomaly")` returns rare objects that belong to none of the six classes.
Compare `clf.anomaly_energy(...)` on them with its values on the test set. Does the
model notice that they are unusual?

---
*FLARE package: `github.com/applecider-ml/flare`, MIT licensed. Data: the ZTF Bright
Transient Survey benchmark shipped in the repository's `benchmark/` folder. This notebook
is part of the FLARE teaching materials.*